In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Hybrid Recommendation System Demo\n",
    "\n",
    "This notebook demonstrates the three core recommendation approaches implemented in this project:\n",
    "1. **Content‑Based Filtering** (using sentence embeddings)\n",
    "2. **Collaborative Filtering** (SVD matrix factorization)\n",
    "3. **Hybrid Model** (weighted combination)\n",
    "\n",
    "We use a small sample dataset to illustrate the pipelines."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Setup and Data Loading\n",
    "\n",
    "We import the necessary modules and create a minimal DataFrame that matches the expected schema (title, description, category, rating, user_id, item_id)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "from src.data.data_adapter import adapt_data\n",
    "from src.model.content_model import ContentRecommender\n",
    "from src.model.collaborative_model import CollaborativeRecommender\n",
    "from src.model.hybrid_model import HybridRecommender"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create a small product catalog with user interaction data\n",
    "products = pd.DataFrame({\n",
    "    'title': ['Wireless Keyboard', 'Bluetooth Speaker', 'Fitness Band', \n",
    "              'Study Lamp', 'Laptop Stand', 'Noise Cancelling Earbuds'],\n",
    "    'description': ['Mechanical keyboard with RGB', 'Portable Bluetooth speaker',\n",
    "                    'Fitness tracker with heart rate', 'LED desk lamp',\n",
    "                    'Aluminium laptop stand', 'Active noise cancelling earbuds'],\n",
    "    'category': ['Accessories', 'Electronics', 'Wearable', \n",
    "                 'Home Office', 'Accessories', 'Electronics'],\n",
    "    'rating': [4.3, 4.5, 4.1, 4.0, 4.2, 4.6],\n",
    "    'item_id': ['i1', 'i2', 'i3', 'i4', 'i5', 'i6']\n",
    "})\n",
    "\n",
    "# Add user-item interactions (ratings)\n",
    "interactions = pd.DataFrame({\n",
    "    'user_id': ['u1','u1','u1','u2','u2','u2','u3','u3','u3','u4','u4','u4'],\n",
    "    'item_id': ['i1','i2','i5','i2','i6','i3','i3','i4','i5','i1','i5','i6'],\n",
    "    'rating': [5,4,4,5,5,3,4,4,5,5,4,5]\n",
    "})\n",
    "\n",
    "# Merge to create a combined DataFrame (as expected by adapt_data)\n",
    "df = products.merge(interactions, on='item_id', how='left')\n",
    "df.head()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Adapt to unified schema (creates 'combined' text feature, fills missing columns)\n",
    "adapted_df, meta = adapt_data(df)\n",
    "print(f\"Adapted DataFrame shape: {adapted_df.shape}\")\n",
    "adapted_df.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Content‑Based Recommendations\n",
    "\n",
    "The `ContentRecommender` computes sentence embeddings for each product’s `combined` text (title + description + category) and finds similar items via cosine similarity."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "content_rec = ContentRecommender(adapted_df)\n",
    "seed = \"Bluetooth Speaker\"\n",
    "print(f\"Content‑based recommendations for '{seed}':\")\n",
    "recs = content_rec.recommend(seed, top_n=3)\n",
    "for r in recs:\n",
    "    print(f\"  - {r['title']} (score: {r['content_score']:.3f})\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Collaborative Filtering\n",
    "\n",
    "The `CollaborativeRecommender` uses user-item ratings to build an interaction matrix and applies Truncated SVD to find latent factors. Recommendations are based on similarity in the latent space."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "collab_rec = CollaborativeRecommender(adapted_df)\n",
    "print(f\"Collaborative recommendations for '{seed}':\")\n",
    "recs = collab_rec.recommend(seed, top_n=3)\n",
    "for r in recs:\n",
    "    score = r.get('collab_score', r.get('score', 0))\n",
    "    print(f\"  - {r['title']} (score: {score:.3f})\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Hybrid Recommendations\n",
    "\n",
    "The `HybridRecommender` combines content and collaborative scores (plus a sentiment component, if available) with adjustable weights α (content), β (collaborative), γ (sentiment). By default, α=0.4, β=0.35, γ=0.25."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "hybrid_rec = HybridRecommender(content_model=content_rec, collab_model=collab_rec)\n",
    "print(f\"Hybrid recommendations for '{seed}':\")\n",
    "recs = hybrid_rec.recommend(seed, top_n=3)\n",
    "for r in recs:\n",
    "    print(f\"  - {r['title']} (hybrid score: {r['hybrid_score']:.3f})\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Customising Weights\n",
    "\n",
    "You can change the influence of each component dynamically."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Give more weight to collaborative filtering\n",
    "hybrid_rec.set_weights(alpha=0.2, beta=0.7, gamma=0.1)\n",
    "print(\"Hybrid recommendations with β=0.7 (collaborative emphasis):\")\n",
    "recs = hybrid_rec.recommend(seed, top_n=3)\n",
    "for r in recs:\n",
    "    print(f\"  - {r['title']} (hybrid score: {r['hybrid_score']:.3f})\")\n",
    "\n",
    "# Restore default weights\n",
    "hybrid_rec.set_weights(alpha=0.4, beta=0.35, gamma=0.25)\n",
    "print(\"\\nBack to default weights:\")\n",
    "recs = hybrid_rec.recommend(seed, top_n=3)\n",
    "for r in recs:\n",
    "    print(f\"  - {r['title']} (hybrid score: {r['hybrid_score']:.3f})\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Conclusion\n",
    "\n",
    "This notebook demonstrates the full recommendation pipeline of the Hybrid Recommender project:\n",
    "- **Content‑based** uses item metadata.\n",
    "- **Collaborative** uses user interaction patterns.\n",
    "- **Hybrid** combines both (and optionally sentiment) for robust recommendations.\n",
    "\n",
    "Users can experiment with different items, datasets, and weights to understand how the models behave."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3 (ipykernel)",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.10.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}